In [1]:
import warnings
warnings.filterwarnings("ignore", message="`langchain-community` is being sunset")

from langchain_community.document_loaders import WikipediaLoader, Docx2txtLoader, PyPDFLoader, TextLoader

In [2]:
from langchain_chroma import Chroma

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
import os

In [4]:
OPENAI_API_KEY = os.environ['OPENAI_API_KEY']

In [5]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)

In [6]:
## embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [7]:
embeddings_model = OpenAIEmbeddings()

In [8]:
vector_db = Chroma("tourist_info", embeddings_model)

In [9]:
def split_and_import(loader):
    chunks = text_splitter.split_documents(loader.load())
    vector_db.add_documents(chunks)
    print(f"Ingested chunks created by {loader}")

In [10]:
import wikipedia
# Descriptive UA avoids Wikimedia's 429 on the lib's shared default UA
wikipedia.wikipedia.USER_AGENT = "tourist-info-notebook/1.0 (mic.a.elle.chlon@gmail.com)"

wikipedia_loader = WikipediaLoader(query="Paestum")
split_and_import(wikipedia_loader)

word_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")
split_and_import(word_loader)

pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
split_and_import(pdf_loader)

txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
split_and_import(txt_loader)

Ingested chunks created by <langchain_community.document_loaders.wikipedia.WikipediaLoader object at 0x7f989d9e8980>
Ingested chunks created by <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x7f989d9e8ec0>
Ingested chunks created by <langchain_community.document_loaders.pdf.PyPDFLoader object at 0x7f989d9ebe00>
Ingested chunks created by <langchain_community.document_loaders.text.TextLoader object at 0x7f989d9eba10>


# 7.2.3/ Ingest multiple documents from a folder

In [12]:
loader_classes = {
    'docx': Docx2txtLoader,
    'pdf': PyPDFLoader,
    'txt': TextLoader,
}

import os

def get_loader(filename):
    _, file_extension = os.path.splitext(filename)
    file_extension = file_extension.lstrip('.')

    loader_class = loader_classes.get(file_extension)

    if loader_class:
        return loader_class(filename)
    else:
        raise ValueError(f"No loader available for file extension '{file_extension}'")

### Ingesting the files from the folder 

In [13]:
folder_path = "ClientoTouristInfo/" # Path to the folder containing the documents

In [14]:
for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename) # Construct the full path to the file
    if os.path.isfile(file_path): # Check if it is a file (not a directory)
        try:
            loader = get_loader(file_path) # Instantiate the correct loader for the file
            print(f"Loader for {filename}: {loader}")
            split_and_import(loader)
        except ValueError as e:
            print(e)

Loader for NationalParkOfCilentoAndValloDiDiano.txt: <langchain_community.document_loaders.text.TextLoader object at 0x7f9875f4c050>
Ingested chunks created by <langchain_community.document_loaders.text.TextLoader object at 0x7f9875f4c050>
Loader for Cilentan coast.docx: <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x7f989db71810>
Ingested chunks created by <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x7f989db71810>
Loader for Cilento Coast Map and Travel Guide.docx: <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x7f9875f4c050>
Ingested chunks created by <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x7f9875f4c050>
Loader for Cilento.docx: <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x7f9875986ea0>
Ingested chunks created by <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x7f9875986ea0>
Loader for M

### Alternative: Ingestive all files with DirectoryLoader